# Urban Pulse NYC - Analyst Tasks 1 to 6 (volume and demand)
Every task reads from MySQL (`urban_pulse_db.service_requests`).
All 118,866 rows are used in these six tasks. The data is a ~3% systematic sample, so **proportions and trends are valid, absolute volumes are not**.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, text

PROJECT = "C:/Users/skmah/urban_pulse"
if os.path.isdir(PROJECT):
    os.chdir(PROJECT)
print("Working folder:", os.getcwd())

os.makedirs("figures", exist_ok=True)
os.makedirs("sql", exist_ok=True)

engine = create_engine("mysql+pymysql://root:@localhost:3306/urban_pulse_db")
pd.options.display.width = 140

QUERIES = {} 

def task(num, title, sql):
    """Run a SQL query, remember it, and return the result as a DataFrame."""
    QUERIES[num] = (title, sql.strip())
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

def save_fig(num, name):
    path = f"figures/task{num:02d}_{name}.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    print("Saved chart:", path)

with engine.connect() as conn:
    TOTAL = int(pd.read_sql(text("SELECT COUNT(*) AS n FROM service_requests"), conn).loc[0, "n"])
print("Connected. Rows in service_requests:", f"{TOTAL:,}")

## Task 1 - Total distinct request volume
**Question:** How many distinct service requests are in the dataset, and what period do they cover?

In [ ]:
df1 = task(1, "Total distinct request volume",
    """SELECT COUNT(DISTINCT unique_key) AS total_distinct_requests,
              MIN(created_date)           AS first_request,
              MAX(created_date)           AS last_request
       FROM service_requests""")
display(df1)
print("Sample size (n):", f"{TOTAL:,}", "| Visualization: KPI number (no chart needed)")

## Task 2 - Top 10 complaint categories by count
**Question:** Which complaint types are reported most often?

In [ ]:
df2 = task(2, "Top 10 complaint categories by count",
    """SELECT complaint_type,
              COUNT(*) AS requests,
              ROUND(100 * COUNT(*) / (SELECT COUNT(*) FROM service_requests), 2) AS pct_of_all
       FROM service_requests
       GROUP BY complaint_type
       ORDER BY requests DESC
       LIMIT 10""")
display(df2)
print("Top 10 categories = ", round(df2["pct_of_all"].sum(), 1), "% of all requests (n =", f"{TOTAL:,})")

plt.figure(figsize=(9, 5))
plt.barh(df2["complaint_type"][::-1], df2["requests"][::-1], color="steelblue")
plt.xlabel("Requests (3% sample)")
plt.title("Task 2: Top 10 complaint categories")
save_fig(2, "top10_complaints")
plt.show()

## Task 3 - Agency workload distribution
**Question:** How is the recorded workload spread across agencies?

In [ ]:
df3 = task(3, "Agency workload distribution",
    """SELECT agency,
              COUNT(*) AS requests,
              ROUND(100 * COUNT(*) / (SELECT COUNT(*) FROM service_requests), 2) AS pct_of_all
       FROM service_requests
       GROUP BY agency
       ORDER BY requests DESC""")
display(df3)
print("Agencies:", len(df3), "| n =", f"{TOTAL:,}")

plt.figure(figsize=(9, 6))
plt.barh(df3["agency"][::-1], df3["requests"][::-1], color="seagreen")
plt.xlabel("Requests (3% sample)")
plt.title("Task 3: Requests by agency")
save_fig(3, "agency_workload")
plt.show()

## Task 4 - Request volume by borough
**Question:** Which boroughs generate the most requests? (`UNSPECIFIED` is kept and labelled, not deleted.)

In [ ]:
df4 = task(4, "Request volume by borough",
    """SELECT borough,
              COUNT(*) AS requests,
              ROUND(100 * COUNT(*) / (SELECT COUNT(*) FROM service_requests), 2) AS pct_of_all
       FROM service_requests
       GROUP BY borough
       ORDER BY requests DESC""")
display(df4)
print("n =", f"{TOTAL:,}", "| Note: counts are raw volume, not per-capita")

plt.figure(figsize=(8, 5))
plt.bar(df4["borough"], df4["requests"], color="darkorange")
plt.ylabel("Requests (3% sample)")
plt.title("Task 4: Requests by borough")
plt.xticks(rotation=20)
save_fig(4, "borough_volume")
plt.show()

## Task 5 - Monthly request trend
**Question:** How does request volume change month to month? (Uses `created_ym`, same sampling fraction every month.)

In [ ]:
df5 = task(5, "Monthly request trend",
    """SELECT created_ym,
              COUNT(*) AS requests
       FROM service_requests
       GROUP BY created_ym
       ORDER BY created_ym""")
display(df5)
print("Months:", len(df5), "| min:", df5["requests"].min(), "| max:", df5["requests"].max(),
      "| mean:", round(df5["requests"].mean(), 1), "| n =", f"{TOTAL:,}")

plt.figure(figsize=(10, 4.5))
plt.plot(df5["created_ym"], df5["requests"], marker="o", color="purple")
plt.ylim(0, df5["requests"].max() * 1.15)
plt.ylabel("Requests (3% sample)")
plt.title("Task 5: Monthly request trend")
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
save_fig(5, "monthly_trend")
plt.show()

## Task 6 - Demand by weekday and hour
**Question:** At which days and hours is demand highest? (Heatmap: darker = more requests.)

In [ ]:
df6 = task(6, "Demand by weekday and hour",
    """SELECT created_weekday,
              created_hour,
              COUNT(*) AS requests
       FROM service_requests
       GROUP BY created_weekday, created_hour""")

order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
grid = df6.pivot(index="created_weekday", columns="created_hour", values="requests").reindex(order).fillna(0)

print("Requests by weekday:")
display(grid.sum(axis=1).astype(int).to_frame("requests"))
print("Top 5 hours of the day (all weekdays combined):")
display(grid.sum(axis=0).astype(int).sort_values(ascending=False).head(5).to_frame("requests"))
print("n =", f"{int(grid.values.sum()):,}")

fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(grid.values, aspect="auto", cmap="Blues")
ax.set_yticks(range(7))
ax.set_yticklabels(order)
ax.set_xticks(range(24))
ax.set_xticklabels(range(24))
ax.set_xlabel("Hour of day (0-23)")
ax.set_title("Task 6: Requests by weekday and hour")
plt.colorbar(im, label="Requests (3% sample)")
save_fig(6, "weekday_hour_heatmap")
plt.show()

## Save all queries from this notebook
Creates `sql/02_tasks_01_06.sql` - one of your required deliverables (SQL queries for the 15 tasks).

In [ ]:
with open("sql/02_tasks_01_06.sql", "w", encoding="utf-8") as f:
    f.write("-- Urban Pulse NYC: analyst tasks 1-6 (run against urban_pulse_db)\n\n")
    for num in sorted(QUERIES):
        title, sql = QUERIES[num]
        f.write(f"-- Task {num}: {title}\n{sql};\n\n")
print("Saved sql/02_tasks_01_06.sql with", len(QUERIES), "queries")